# 01 · UNSW-NB15 — Exploratory Data Analysis

This notebook performs a thorough EDA of the **UNSW-NB15** dataset  
(partitioned train/test version: 175 341 train + 82 332 test = **257 673** records).

### Outline
1. Load & concatenate train + test sets
2. Schema overview — dtype, missing values, unique counts
3. Class / attack-category distribution
4. Missing & infinite value report
5. Duplicate record count
6. Categorical vs numerical feature split
7. Distribution summary for every feature
8. Correlation with the label
9. Correlation heatmap
10. Per-attack-type feature distributions (top-5 discriminative features)
11. Potential label-leakage features
12. Key findings & takeaways

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:.4f}".format)

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="viridis",
    font_scale=1.05,
)

%matplotlib inline

---
## 1 · Load Data

In [ ]:
RAW = Path("..") / "data" / "raw"

train = pd.read_csv(RAW / "UNSW_NB15_training-set.csv")
test  = pd.read_csv(RAW / "UNSW_NB15_testing-set.csv")

print(f"Train shape: {train.shape}")
print(f"Test  shape: {test.shape}")

# Tag source before merging
train["_split"] = "train"
test["_split"]  = "test"
df = pd.concat([train, test], ignore_index=True)
print(f"\nCombined shape: {df.shape}")
df.head(3)

In [ ]:
# Drop the 'id' column if present — it is just a row index
if "id" in df.columns:
    df = df.drop(columns=["id"])
    print("Dropped 'id' column.")

# Standardise attack_cat values (strip whitespace, title-case)
if "attack_cat" in df.columns:
    df["attack_cat"] = df["attack_cat"].astype(str).str.strip().str.title()
    # Some versions store blanks / NaN for Normal traffic
    df.loc[df["attack_cat"].isin(["Nan", "", "None", "Normal "]), "attack_cat"] = "Normal"
    df.loc[df["label"] == 0, "attack_cat"] = "Normal"
    print("Standardised attack_cat values.")

print(f"Working shape: {df.shape}")

---
## 2 · Schema Overview

For **every feature** we report:
- dtype
- missing value count & percentage
- unique value count
- distribution summary (mean / std / min / 25% / 50% / 75% / max for numeric; top value & frequency for categorical)

In [ ]:
# Exclude helper columns from analysis
analysis_cols = [c for c in df.columns if c != "_split"]

schema = pd.DataFrame({
    "dtype":          df[analysis_cols].dtypes,
    "missing_count":  df[analysis_cols].isnull().sum(),
    "missing_pct":    (df[analysis_cols].isnull().mean() * 100).round(2),
    "unique_count":   df[analysis_cols].nunique(),
})

schema

In [ ]:
# Full distribution summary (numeric columns)
num_cols = df[analysis_cols].select_dtypes(include="number").columns.tolist()
cat_cols = df[analysis_cols].select_dtypes(exclude="number").columns.tolist()

print(f"Numerical features : {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")
print(f"\nCategorical: {cat_cols}")

In [ ]:
# Numeric summary
df[num_cols].describe().T

In [ ]:
# Categorical summary
for col in cat_cols:
    vc = df[col].value_counts(dropna=False)
    print(f"\n{'─'*40}")
    print(f"Feature: {col}  ({df[col].dtype})")
    print(f"  Unique values: {df[col].nunique()}")
    print(f"  Missing:       {df[col].isnull().sum()}")
    print(f"  Top-10 values:")
    print(vc.head(10).to_string())

---
## 3 · Class Distribution (`attack_cat`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── 3a. Binary label distribution ────────────────────────────────────
label_counts = df["label"].value_counts().sort_index()
colors_bin = ["#2ecc71", "#e74c3c"]
axes[0].bar(
    ["Normal (0)", "Attack (1)"],
    label_counts.values,
    color=colors_bin,
    edgecolor="black",
    linewidth=0.5,
)
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 1000, f"{v:,}\n({v/len(df)*100:.1f}%)",
                ha="center", fontweight="bold")
axes[0].set_title("Binary Label Distribution", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Count")

# ── 3b. Attack category distribution ─────────────────────────────────
cat_counts = df["attack_cat"].value_counts()
palette = sns.color_palette("Set2", n_colors=len(cat_counts))
bars = axes[1].barh(cat_counts.index[::-1], cat_counts.values[::-1],
                    color=palette, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, cat_counts.values[::-1]):
    axes[1].text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
                f"{val:,}", va="center", fontsize=9)
axes[1].set_title("Attack Category Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.savefig("../results/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Train vs Test split per attack category
split_counts = df.groupby(["attack_cat", "_split"]).size().unstack(fill_value=0)
split_counts["total"] = split_counts.sum(axis=1)
split_counts = split_counts.sort_values("total", ascending=False)
split_counts

---
## 4 · Missing & Infinite Value Report

In [ ]:
# ── Missing values ───────────────────────────────────────────────────
missing = df[analysis_cols].isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) == 0:
    print("✅  No missing values found in any column.")
else:
    print(f"⚠️  {len(missing)} column(s) with missing values:\n")
    for col, cnt in missing.items():
        print(f"    {col:30s}  {cnt:>8,}  ({cnt/len(df)*100:.2f}%)")

In [ ]:
# ── Infinite values (numeric columns only) ───────────────────────────
inf_report = {}
for col in num_cols:
    n_inf = np.isinf(df[col].values).sum() if df[col].dtype != object else 0
    if n_inf > 0:
        inf_report[col] = n_inf

if not inf_report:
    print("✅  No infinite values found in any numeric column.")
else:
    print(f"⚠️  {len(inf_report)} column(s) contain infinite values:\n")
    for col, cnt in sorted(inf_report.items(), key=lambda x: -x[1]):
        print(f"    {col:30s}  {cnt:>8,}  ({cnt/len(df)*100:.2f}%)")

---
## 5 · Duplicate Records

In [ ]:
feature_cols = [c for c in analysis_cols if c not in ("label", "attack_cat")]

n_dup_all    = df.duplicated(subset=analysis_cols).sum()
n_dup_feats  = df.duplicated(subset=feature_cols).sum()

print(f"Exact duplicate rows (all columns):         {n_dup_all:>8,}  ({n_dup_all/len(df)*100:.2f}%)")
print(f"Duplicate feature vectors (ignoring labels): {n_dup_feats:>8,}  ({n_dup_feats/len(df)*100:.2f}%)")

if n_dup_feats > n_dup_all:
    print("\n⚠️  Some rows share identical features but different labels — ")
    print("   this indicates inherent label noise in the dataset.")

---
## 6 · Categorical vs Numerical Feature Split

In [ ]:
# Refine: some numeric columns may actually be binary / ordinal
binary_cols = [c for c in num_cols if df[c].dropna().nunique() <= 2]
continuous_cols = [c for c in num_cols if c not in binary_cols]

print(f"{'Category':<25s} {'Count':>5s}")
print("─" * 35)
print(f"{'Continuous numeric':<25s} {len(continuous_cols):>5d}")
print(f"{'Binary numeric':<25s} {len(binary_cols):>5d}")
print(f"{'Categorical (object)':<25s} {len(cat_cols):>5d}")
print("─" * 35)
print(f"{'TOTAL':<25s} {len(analysis_cols):>5d}")

if binary_cols:
    print(f"\nBinary columns: {binary_cols}")

---
## 7 · Per-Feature Distribution Summary

Quick histogram grid for numeric features.

In [ ]:
plot_num = [c for c in num_cols if c != "label"]
n = len(plot_num)
ncols = 5
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 2.8))
axes = axes.flatten()

for i, col in enumerate(plot_num):
    ax = axes[i]
    data = df[col].replace([np.inf, -np.inf], np.nan).dropna()
    # Clip extreme outliers for visualisation
    lo, hi = data.quantile(0.01), data.quantile(0.99)
    data_clipped = data.clip(lo, hi)
    ax.hist(data_clipped, bins=50, color="steelblue", edgecolor="white", linewidth=0.3)
    ax.set_title(col, fontsize=9, fontweight="bold")
    ax.tick_params(labelsize=7)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Numeric Feature Distributions (1st–99th percentile)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../results/feature_distributions.png", dpi=120, bbox_inches="tight")
plt.show()

---
## 8 · Correlation with the Label

We compute **point-biserial / Pearson correlation** between each numeric feature and the binary `label` column, and rank features by absolute correlation.

In [ ]:
label_corr = (
    df[num_cols]
    .replace([np.inf, -np.inf], np.nan)
    .corrwith(df["label"])
    .drop("label", errors="ignore")
    .dropna()
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(10, max(6, len(label_corr) * 0.25)))
colors = ["#e74c3c" if v > 0 else "#3498db" for v in label_corr.values]
ax.barh(label_corr.index[::-1], label_corr.values[::-1], color=colors[::-1],
        edgecolor="white", linewidth=0.3)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Pearson Correlation with label")
ax.set_title("Feature Correlation with Binary Label", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/label_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTop 15 most correlated features (absolute value):")
label_corr.head(15)

---
## 9 · Correlation Heatmap

Full numeric feature correlation matrix.

In [ ]:
corr_matrix = (
    df[num_cols]
    .replace([np.inf, -np.inf], np.nan)
    .corr()
)

fig, ax = plt.subplots(figsize=(18, 15))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.3,
    annot=False,
    cbar_kws={"shrink": 0.7, "label": "Pearson r"},
    ax=ax,
)
ax.set_title("Numeric Feature Correlation Heatmap", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Highly correlated feature pairs (|r| > 0.90)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.90:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                round(r, 3),
            ))

hc_df = pd.DataFrame(high_corr_pairs, columns=["Feature A", "Feature B", "Pearson r"])
hc_df = hc_df.sort_values("Pearson r", key=abs, ascending=False).reset_index(drop=True)
print(f"Feature pairs with |r| > 0.90 : {len(hc_df)}\n")
hc_df

---
## 10 · Per-Attack-Type Feature Distributions

### 10a. Identify the 5 most discriminative features

We use ANOVA F-statistic across attack categories to rank features by discriminative power.

In [ ]:
# One-way ANOVA across attack categories for each numeric feature
f_scores = {}
attack_groups = df.groupby("attack_cat")

for col in num_cols:
    if col == "label":
        continue
    groups = [grp[col].replace([np.inf, -np.inf], np.nan).dropna().values
              for _, grp in attack_groups]
    # Drop groups with < 2 observations
    groups = [g for g in groups if len(g) >= 2]
    if len(groups) >= 2:
        f_stat, p_val = stats.f_oneway(*groups)
        f_scores[col] = f_stat

f_rank = pd.Series(f_scores).sort_values(ascending=False)
top5 = f_rank.head(5).index.tolist()

print("Top-5 most discriminative features (ANOVA F-statistic):\n")
for rank, (feat, f_val) in enumerate(f_rank.head(5).items(), 1):
    print(f"  {rank}. {feat:30s}  F = {f_val:>12,.1f}")

### 10b. Box-plots: top-5 features × attack category

In [ ]:
fig, axes = plt.subplots(len(top5), 1, figsize=(14, 5 * len(top5)))

attack_order = df["attack_cat"].value_counts().index.tolist()

for idx, feat in enumerate(top5):
    ax = axes[idx]
    plot_data = df[[feat, "attack_cat"]].copy()
    plot_data[feat] = plot_data[feat].replace([np.inf, -np.inf], np.nan)
    # Clip to 1st–99th percentile for readable plots
    lo, hi = plot_data[feat].quantile(0.01), plot_data[feat].quantile(0.99)
    plot_data[feat] = plot_data[feat].clip(lo, hi)

    sns.boxplot(
        data=plot_data,
        x="attack_cat",
        y=feat,
        order=attack_order,
        palette="Set2",
        ax=ax,
        fliersize=1,
    )
    ax.set_title(f"{feat}  (rank #{idx+1})", fontsize=12, fontweight="bold")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)

plt.suptitle("Top-5 Discriminative Features by Attack Category",
             fontsize=15, fontweight="bold", y=1.005)
plt.tight_layout()
plt.savefig("../results/top5_features_by_attack.png", dpi=150, bbox_inches="tight")
plt.show()

### 10c. Pairwise KDE: selected attacks vs Normal

For each of the top-5 features, compare the distribution in **DoS**, **Reconnaissance**, **Exploits** vs **Normal**.

In [ ]:
compare_cats = ["Dos", "Reconnaissance", "Exploits"]
# Normalise category names to match what's in the data
available_cats = df["attack_cat"].unique().tolist()
compare_cats = [c for c in compare_cats if c in available_cats]
# Fallback: if DoS is stored differently
for alt in ["DoS", "dos", "DOS"]:
    if alt in available_cats and alt not in compare_cats:
        compare_cats.insert(0, alt)
        break

print(f"Comparing categories: {compare_cats}")

fig, axes = plt.subplots(len(top5), len(compare_cats),
                         figsize=(5 * len(compare_cats), 4 * len(top5)),
                         squeeze=False)

for i, feat in enumerate(top5):
    normal_data = df.loc[df["attack_cat"] == "Normal", feat]
    normal_data = normal_data.replace([np.inf, -np.inf], np.nan).dropna()
    lo, hi = normal_data.quantile(0.01), normal_data.quantile(0.99)
    normal_clipped = normal_data.clip(lo, hi)

    for j, cat in enumerate(compare_cats):
        ax = axes[i][j]
        attack_data = df.loc[df["attack_cat"] == cat, feat]
        attack_data = attack_data.replace([np.inf, -np.inf], np.nan).dropna()
        attack_clipped = attack_data.clip(lo, hi)

        ax.hist(normal_clipped, bins=60, density=True, alpha=0.5,
                color="#2ecc71", label="Normal")
        ax.hist(attack_clipped, bins=60, density=True, alpha=0.5,
                color="#e74c3c", label=cat)
        ax.set_title(f"{feat}\n{cat} vs Normal", fontsize=10, fontweight="bold")
        ax.legend(fontsize=8)
        ax.tick_params(labelsize=7)

plt.suptitle("Per-Attack Feature Distributions (top-5 features)",
             fontsize=15, fontweight="bold", y=1.005)
plt.tight_layout()
plt.savefig("../results/attack_vs_normal_kde.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 11 · Potential Label-Leakage Features

Features that could **leak the label** are those that:
1. Are derived from the label itself (tautological predictors)
2. Have **suspiciously high** correlation with the label (|r| > 0.85)
3. Have near-perfect separation of classes

We flag any features matching these criteria.

In [ ]:
LEAK_THRESHOLD = 0.85

# Known leakage candidates in UNSW-NB15
known_suspects = {
    "attack_cat": "Directly encodes the attack type (categorical label).",
    "ct_state_ttl": (
        "Highly correlated with label in many studies — counts of "
        "connections by state/TTL may implicitly encode attack signatures."
    ),
}

leak_candidates = []

# Check numeric features
for feat, corr_val in label_corr.items():
    if abs(corr_val) >= LEAK_THRESHOLD:
        leak_candidates.append({
            "feature": feat,
            "reason": f"|r| = {abs(corr_val):.3f}  (above {LEAK_THRESHOLD} threshold)",
        })

# Check known suspects
for feat, reason in known_suspects.items():
    if feat in df.columns and feat not in [lc["feature"] for lc in leak_candidates]:
        leak_candidates.append({"feature": feat, "reason": reason})

if leak_candidates:
    print("⚠️  POTENTIAL LABEL-LEAKAGE FEATURES\n")
    for lc in leak_candidates:
        print(f"  • {lc['feature']:30s}  → {lc['reason']}")
else:
    print("✅  No features exceed the leakage correlation threshold.")

print("\n📌  Recommendation: drop 'attack_cat' from feature inputs and")
print("    carefully evaluate features with |r| > 0.7 for causal leakage.")

---
## 12 · Key Findings & Takeaways

### Dataset Overview
- **257,673** records (175,341 train + 82,332 test), **~45 columns** including `label` and `attack_cat`.
- **10 classes**: Normal + 9 attack categories (Generic, Exploits, Fuzzers, DoS, Reconnaissance, Analysis, Backdoor, Shellcode, Worms).
- The dataset is **moderately imbalanced** — Normal traffic is roughly 1/3 of records; Generic and Exploits dominate the attack side.

### Data Quality
- **Missing values**: `attack_cat` may have blanks/NaN for normal samples; a few features (e.g., `ct_ftp_cmd`, `is_ftp_login`) can have missing entries.
- **Infinite values**: Likely absent, but some rate-based features (e.g., `sload`, `dload`) should be checked.
- **Duplicates**: Duplicate feature vectors with different labels indicate inherent label noise.

### Feature Characteristics
- Most features are **numerical / continuous** (flow bytes, packet counts, timing).
- **Categorical** features: `proto`, `state`, `service`, `attack_cat`.
- Several feature pairs are **highly correlated** (|r| > 0.9) — candidates for dimensionality reduction.

### Leakage Risks
- **`attack_cat`** must be excluded from features — it *is* the label.
- **`ct_state_ttl`** and similar connection-tracking features often have suspiciously high correlation with `label`. These should be evaluated carefully; they may be legitimate features *or* artifacts of the data generation process.
- Any feature with |r| > 0.85 warrants manual inspection before inclusion in a production model.

### Recommendations for Modeling
1. **Drop** `id` (row index) and `attack_cat` from input features.
2. **Encode** categorical features (`proto`, `state`, `service`) via one-hot or target encoding.
3. **Handle** missing values (impute or flag) and infinite values (clip or replace).
4. **Consider** removing one feature from highly correlated pairs (|r| > 0.95) to reduce multicollinearity.
5. **Evaluate** `ct_state_ttl` and top-correlated features for potential leakage before finalising the feature set.